# Propagate Inside Run

Clean Marmousi acoustic propagation run. This notebook uses a reduced Marmousi velocity model, compares a known 3x3 FD baseline with the numerical OPT operator, and keeps plotting/video steps separate from construction.


## 1. Setup


In [ ]:
using Pkg
cd(@__DIR__)
Pkg.activate("../")

try
    using Metal
catch err
    @warn "Metal not loaded; continuing on CPU" err
end

using JLD2
ParamFile = "../config/testparam.csv"

include("../src/batchFiles/batchGPU.jl")
include("../src/commonBatchs.jl")
include("../src/planet1D.jl")
include("../src/GeoPoints.jl")
using .commonBatchs, .planet1D, .GeoPoints

include("../src/flexOPT.jl")
using .flexOPT

include("temporaryHelpers.jl")


## 2. Reduced Marmousi Model


In [ ]:
marmousi = load(joinpath(@__DIR__, "tmp/seismicModelMarmousi.jld2"), "output")

shape = (151, 151)
downsample_step = 4
velocity = downsample_center_crop(marmousi.Vpv .* 1e3, shape; step=downsample_step)

velocity_value = median(vec(velocity))
dx = 1.0e3 * downsample_step
cfl = 0.35
cfl_info = cfl_diagnostics(velocity, (dx, dx, 1.0); cfl_safety=cfl)
dt = cfl_info.suggested_dt_2D
delta = (dx, dx, dt)

sourcePoint = CartesianIndex(cld(size(velocity, 1), 2), cld(size(velocity, 2), 2))
Nt = 120
store_every = 3
total_time = Nt * dt

# Keep the Ricker packet inside this short debug run.
source_f0 = min(max(cfl_info.suggested_f0, 0.08), 1 / (8dt))
source_t0 = min(12dt, 0.25total_time)
source_amplitude = 1.0

@show size(velocity) extrema(velocity) velocity_value delta sourcePoint Nt store_every total_time source_f0 source_t0


## 3. FD Baseline Point Source


In [ ]:
preparedFD = prepare_fd2d_acoustic_baseline(velocity, delta)
timeSignalFD = source_time_samples(Nt, dt, preparedFD.timePointsUsedForOneStep; t0=source_t0, f0=source_f0)
sourceFD = point_source_full(preparedFD, sourcePoint, timeSignalFD; iForceField=1, amplitude=source_amplitude)
source_peak_index_fd = argmax(abs.(timeSignalFD))
source_peak_time_fd = (source_peak_index_fd - 1) * dt
@show maximum(abs, timeSignalFD) source_peak_index_fd source_peak_time_fd maximum(abs, sourceFD)

frames_fd_full = propagate_linear_frames_with_source(
    preparedFD,
    Nt;
    sourceFull=sourceFD,
    store_every=store_every,
    blowup_limit=1e12,
)
frames_fd = component_frames(frames_fd_full, 1)
fd_report = wavefield_snapshot_report(frames_fd)
fd_drift = drift_report(frames_fd, sourcePoint)

@show length(frames_fd) fd_report[1] fd_report[end]
@show fd_drift[end]


In [ ]:
fig_fd = plot_wave_snapshots(
    frames_fd;
    sourcePoint=sourcePoint,
    title="FD acoustic Marmousi reduced point source",
)
fig_fd


## 4. OPT Acoustic Point Source


In [ ]:
# Build the OPT recipe in dimensionless grid coordinates.
# With physical Δ=(4000 m, 4000 m, dt), the Taylor inversion is badly scaled and
# can collapse to an almost pure temporal stencil.  In unit grid coordinates,
# the material variable is the local CFL velocity v*dt/dx.
@assert isapprox(delta[1], delta[2]; rtol=1e-12, atol=0.0)
velocity_cfl = velocity .* (dt / delta[1])
delta_opt_recipe = (1.0, 1.0, 1.0)

toyOPT = build_opt_prepared(
    "2DacousticTime",
    [velocity_cfl],
    delta_opt_recipe;
    pointsInSpace=3,
    pointsInTime=3,
    supplementaryOrder=2,
    orderBspace=1,
    orderBtime=1,
    YorderBspace=-1,
    YorderBtime=-1,
    modelName="Marmousi_acoustic_reduced_OPT_dimensionless",
)
preparedOPT = toyOPT.prepared
opt_A_report = implicit_matrix_report(preparedOPT)
st_opt = operator_stencil_at_point(toyOPT.numOps, sourcePoint; which=:left)
st_opt_summary = stencil_time_summary(st_opt)
st_opt_matrices = stencil_matrices_by_time(st_opt)
st_opt_vs_fd = compare_stencil_scale_to_fd(st_opt, velocity_cfl[sourcePoint], 1.0, 1.0)

@show extrema(velocity_cfl) velocity_cfl[sourcePoint]
@show preparedOPT.spaceShape preparedOPT.NpointsSpace preparedOPT.NField preparedOPT.NForceField
@show opt_A_report
st_opt_summary, st_opt_matrices, st_opt_vs_fd


## OPT RHS Source Diagnostics

`sourceFull` is the physical source field samples. The operator `Γ` should distribute that source over neighboring equations in space and time.

In [ ]:
st_rhs = rhs_stencil_at_source(toyOPT.numOps, sourcePoint; iExpr=1, iForceField=1)
rhs_summary = stencil_time_summary(st_rhs)
rhs_matrices = stencil_matrices_by_time(st_rhs)
rhs_summary, rhs_matrices


In [ ]:
timeSignalOPT = source_time_samples(Nt, dt, preparedOPT.timePointsUsedForOneStep; t0=source_t0, f0=source_f0)
sourceOPT = point_source_full(preparedOPT, sourcePoint, timeSignalOPT; iForceField=1, amplitude=source_amplitude)
source_peak_index_opt = argmax(abs.(timeSignalOPT))
source_peak_time_opt = (source_peak_index_opt - 1) * dt
@show maximum(abs, timeSignalOPT) source_peak_index_opt source_peak_time_opt maximum(abs, sourceOPT)

source_rhs_rows_opt = source_rhs_scan(preparedOPT, sourceOPT; its=1:min(12, Nt))
source_rhs_peak_it_opt = min(source_peak_index_opt, Nt)
source_rhs_peak_opt = source_rhs_diagnostics(preparedOPT, sourceOPT; it=source_rhs_peak_it_opt)
@show source_rhs_rows_opt
@show source_rhs_peak_opt.it source_rhs_peak_opt.force_max source_rhs_peak_opt.b_max source_rhs_peak_opt.b_norm source_rhs_peak_opt.b_argmax

frames_opt_full = propagate_linear_frames_with_source(
    preparedOPT,
    Nt;
    sourceFull=sourceOPT,
    store_every=store_every,
    blowup_limit=1e12,
)
frames_opt = component_frames(frames_opt_full, 1)
opt_report = wavefield_snapshot_report(frames_opt)
opt_drift = drift_report(frames_opt, sourcePoint)

@show length(frames_opt) opt_report[1] opt_report[end]
@show opt_drift[end]


In [ ]:
fig_opt = plot_wave_snapshots(
    frames_opt;
    sourcePoint=sourcePoint,
    title="OPT acoustic Marmousi reduced point source, Y=-1",
)
fig_opt


## 5. Optional Video


In [ ]:
# Run this only after snapshots look reasonable.
# videoFile = joinpath(@__DIR__, "marmousi_acoustic_OPT_point.mp4")
# record_wave_video(frames_opt; videoFile=videoFile, background=velocity, sourcePoint=sourcePoint, clim=nothing)
# videoFile
